# Binary Classification: Heart Disease Prediction

Logistische Regression für `target`: **0 = keine Erkrankung**, **1 = Erkrankung**.
Die bereinigten Daten stammen aus `data_cleaning.ipynb`. Alle Zellen von oben nach unten ausführen; das Notebook benötigt keine Variablen aus einem anderen Notebook.

Fehlende Kategorien werden als `np.nan` an scikit-learn übergeben. Imputation, Skalierung und One-Hot-Encoding sind Teil der Modell-Pipeline und werden ausschließlich auf den Trainingsdaten gelernt. `id`, `dataset` und `num` werden nicht als Eingabemerkmale verwendet.


In [29]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
numeric_columns = ["age", "trestbps", "chol", "thalch", "oldpeak"]
categorical_columns = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
feature_columns = numeric_columns + categorical_columns

ROOT = Path.cwd()
if not (ROOT / "data/processed/heart_disease_clean.csv").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/processed/heart_disease_clean.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Zuerst data_cleaning.ipynb im Projektverzeichnis ausführen.")

# Auch ca und die booleschen Angaben konsistent als Kategorien einlesen.
df = pd.read_csv(DATA_PATH, dtype={column: "string" for column in categorical_columns})
missing_columns = set(feature_columns + ["target"]) - set(df.columns)
if missing_columns:
    raise ValueError(f"Benötigte Spalten fehlen: {sorted(missing_columns)}")

X = df[feature_columns].copy()
for column in categorical_columns:
    # Nach der String-Konvertierung pd.NA durch np.nan ersetzen, nicht davor.
    X[column] = X[column].astype(object).where(X[column].notna(), np.nan)
y = df["target"].copy()
if y.isna().any() or not y.isin([0, 1]).all() or y.nunique() != 2:
    raise ValueError("target muss beide Klassen 0 und 1 enthalten und vollständig sein.")
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training: {len(X_train)} Datensätze; Test: {len(X_test)} Datensätze")
display(pd.DataFrame({"Training": y_train.value_counts(), "Test": y_test.value_counts()}).sort_index())


Training: 734 Datensätze; Test: 184 Datensätze


,Training,Test
target,,
0,328,82
1,406,102


## Vorverarbeitung und Modell trainieren
Die Pipeline erhält die bereinigten, noch nicht kodierten Merkmale und führt die gesamte Vorverarbeitung bei `fit`, `predict` und `predict_proba` automatisch aus.


In [30]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing", keep_empty_features=True)),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
print("Modell erfolgreich trainiert.")


Modell erfolgreich trainiert.


## Testdaten auswerten
Die Confusion Matrix zeigt die tatsächlichen Klassen in den Zeilen und die vorhergesagten Klassen in den Spalten. Die Wahrscheinlichkeiten sind Modellschätzungen. Die Testdaten dienen der abschließenden Bewertung, nicht der Auswahl von Modellparametern.


In [31]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["Keine Erkrankung", "Erkrankung"], zero_division=0,
))
print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["Tatsächlich 0", "Tatsächlich 1"],
    columns=["Vorhergesagt 0", "Vorhergesagt 1"],
))
print(f"ROC-AUC: {roc_auc_score(y_test, y_probability):.3f}")

predictions = pd.DataFrame({
    "target": y_test,
    "prediction": y_pred,
    "probability_heart_disease": y_probability,
})
display(predictions.head(10))


Classification Report:
                  precision    recall  f1-score   support

Keine Erkrankung       0.87      0.82      0.84        82
      Erkrankung       0.86      0.90      0.88       102

        accuracy                           0.86       184
       macro avg       0.86      0.86      0.86       184
    weighted avg       0.86      0.86      0.86       184

Confusion Matrix:


,Vorhergesagt 0,Vorhergesagt 1
Tatsächlich 0,67,15
Tatsächlich 1,10,92


ROC-AUC: 0.933


,target,prediction,probability_heart_disease
513,1,0,0.300502
824,1,1,0.966021
851,1,1,0.627008
803,1,1,0.910333
886,0,1,0.623483
483,0,0,0.284528
273,0,0,0.241917
539,1,1,0.972858
402,0,0,0.110858
600,1,1,0.611256


In [32]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Parametername: Pipeline-Schritt "classifier" + "__" + Parameter "C"
param_grid = {"classifier__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

search = GridSearchCV(
    estimator=clone(model),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

# Untransformierte Trainingsdaten verwenden!
search.fit(X_train, y_train)

print("Beste Parameter:", search.best_params_)
print(f"Beste mittlere CV-ROC-AUC: {search.best_score_:.3f}")

# Ergebnisse aller getesteten C-Werte anzeigen
results = pd.DataFrame(search.cv_results_)
display(
    results[["param_classifier__C", "mean_test_score", "std_test_score"]].sort_values(
        "mean_test_score", ascending=False
    )
)

# Bereits mit den besten Parametern auf allen Trainingsdaten trainiert
best_model = search.best_estimator_


Beste Parameter: {'classifier__C': 0.1}
Beste mittlere CV-ROC-AUC: 0.890


,param_classifier__C,mean_test_score,std_test_score
2,0.100,0.889904,0.047726
3,1.000,0.888179,0.046655
4,10.000,0.885363,0.048542
6,1000.000,0.884729,0.049126
5,100.000,0.884616,0.049120
1,0.010,0.878526,0.052531
0,0.001,0.862045,0.058206


In [33]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

fn_rate = fn / (fn + tp) if fn + tp else float("nan")
negative_error = fn / (tn + fn) if tn + fn else float("nan")

print(f"Übersehene Erkrankte: {fn} von {fn + tp}")
print(f"False-Negative-Rate: {fn_rate:.1%}")
print(f"Fehlalarme: {fp} von {tn + fp} Nicht-Erkrankten")
print(f"Unter den negativen Vorhersagen erkrankt: {negative_error:.1%}")


Übersehene Erkrankte: 10 von 102
False-Negative-Rate: 9.8%
Fehlalarme: 15 von 82 Nicht-Erkrankten
Unter den negativen Vorhersagen erkrankt: 13.0%


In [34]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split

# Falls du Grid Search durchgeführt hast: model durch best_model ersetzen.
threshold_model = clone(model)

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    stratify=y_train,
    random_state=42,
)

threshold_model.fit(X_fit, y_fit)
probabilities = threshold_model.predict_proba(X_val)[:, 1]

rows = []
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, predicted, labels=[0, 1]).ravel()

    rows.append(
        {
            "Schwelle": threshold,
            "Übersehene Erkrankte": fn,
            "False-Negative-Rate": fn / (fn + tp),
            "False-Positive-Rate": fp / (fp + tn),
            "Recall": tp / (tp + fn),
            "Fehlalarmrate": fp / (fp + tn),
        }
    )

display(pd.DataFrame(rows))


,Schwelle,Übersehene Erkrankte,False-Negative-Rate,False-Positive-Rate,Recall,Fehlalarmrate
0,0.1,2,0.019608,0.609756,0.980392,0.609756
1,0.2,3,0.029412,0.451220,0.970588,0.451220
2,0.3,6,0.058824,0.341463,0.941176,0.341463
3,0.4,8,0.078431,0.268293,0.921569,0.268293
4,0.5,15,0.147059,0.243902,0.852941,0.243902
5,0.6,17,0.166667,0.207317,0.833333,0.207317
6,0.7,22,0.215686,0.146341,0.784314,0.146341
7,0.8,33,0.323529,0.109756,0.676471,0.109756
8,0.9,47,0.460784,0.036585,0.539216,0.036585


In [35]:
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

candidates = {
    "Logistische Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200,
        max_leaf_nodes=15,
        learning_rate=0.05,
        random_state=42,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, classifier in candidates.items():
    pipeline = Pipeline(
        [
            ("preprocessing", clone(preprocessor)),
            ("classifier", classifier),
        ]
    )

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
    )

    results.append(
        {
            "Modell": name,
            "Mittlere ROC-AUC": scores.mean(),
            "Standardabweichung": scores.std(),
        }
    )

display(pd.DataFrame(results).sort_values("Mittlere ROC-AUC", ascending=False))


,Modell,Mittlere ROC-AUC,Standardabweichung
0,Logistische Regression,0.888179,0.046655
1,Random Forest,0.885878,0.046348
2,Gradient Boosting,0.869748,0.052192


## Random Forest optimieren

Die Parametersuche läuft ausschließlich innerhalb der Trainingsdaten. Davon bleiben 25 % als separate Validierung für die Entscheidungsschwelle zurück. Die Testdaten werden erst nach Parameter- und Schwellenwahl ausgewertet.

`RandomizedSearchCV` prüft 24 Parameterkombinationen mit jeweils fünf geschichteten Folds. Optimiert wird ROC-AUC; zusätzlich werden Average Precision und Recall bei der Standardschwelle erfasst. Der beste Suchwert ist eine Auswahlkennzahl und kann optimistisch sein.

**Ziel für das Experiment:** mindestens 95 % Recall auf der separaten Validierung, darunter möglichst wenige Fehlalarme. `RECALL_TARGET` ist konfigurierbar und keine klinisch validierte Vorgabe. Die ursprüngliche Random-Forest-Konfiguration wird auf denselben Daten und mit eigener Schwelle verglichen.


In [36]:
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from joblib import parallel_backend

RECALL_TARGET = 0.95
X_rf_fit, X_rf_val, y_rf_fit, y_rf_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)
rf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_baseline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1
    )),
])
rf_param_distributions = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [None, 4, 6, 10, 16],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 3, 5, 10],
    "classifier__max_features": ["sqrt", 0.5, 1.0],
    "classifier__class_weight": [None, "balanced"],
}
rf_search = RandomizedSearchCV(
    estimator=clone(rf_baseline),
    param_distributions=rf_param_distributions,
    n_iter=24,
    scoring={"roc_auc": "roc_auc", "average_precision": "average_precision", "recall": "recall"},
    refit="roc_auc", cv=rf_cv, random_state=RANDOM_STATE,
    n_jobs=2, pre_dispatch=2, error_score="raise",
)
# Zwei Threads: keine verschachtelte Parallelisierung einzelner Wälder.
with parallel_backend("threading", n_jobs=2):
    baseline_cv_scores = cross_val_score(
        clone(rf_baseline), X_rf_fit, y_rf_fit, cv=rf_cv, scoring="roc_auc", n_jobs=2
    )
    rf_search.fit(X_rf_fit, y_rf_fit)

print(f"Baseline CV-ROC-AUC: {baseline_cv_scores.mean():.3f} ± {baseline_cv_scores.std():.3f}")
print(f"Beste Such-CV-ROC-AUC: {rf_search.best_score_:.3f}")
print("Beste Suchparameter:", rf_search.best_params_)
rf_search_results = pd.DataFrame(rf_search.cv_results_).sort_values("rank_test_roc_auc")
display(rf_search_results[[
    "rank_test_roc_auc", "mean_test_roc_auc", "std_test_roc_auc",
    "mean_test_average_precision", "mean_test_recall", "params",
]].head(10))

# Falls die Suche die Baseline nicht übertrifft, bleibt die Baseline bestehen.
rf_use_search = rf_search.best_score_ > baseline_cv_scores.mean()
rf_selected_parameters = rf_search.best_params_ if rf_use_search else {}
rf_validation_model = clone(rf_baseline).set_params(**rf_selected_parameters)
rf_validation_model.fit(X_rf_fit, y_rf_fit)
rf_baseline_validation = clone(rf_baseline).fit(X_rf_fit, y_rf_fit)
print("Ausgewählte Konfiguration:", "Parametersuche" if rf_use_search else "Baseline")


Baseline CV-ROC-AUC: 0.887 ± 0.012
Beste Such-CV-ROC-AUC: 0.888
Beste Suchparameter: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 3, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 6, 'classifier__class_weight': None}


,rank_test_roc_auc,mean_test_roc_auc,std_test_roc_auc,mean_test_average_precision,mean_test_recall,params
0,1,0.888344,0.011810,0.885166,0.852077,"{'classifier__n_estimators': 300, 'classifier_..."
6,2,0.887808,0.010380,0.888871,0.855410,"{'classifier__n_estimators': 200, 'classifier_..."
15,3,0.886136,0.008905,0.888269,0.852077,"{'classifier__n_estimators': 300, 'classifier_..."
14,4,0.885276,0.014826,0.882336,0.828962,"{'classifier__n_estimators': 200, 'classifier_..."
10,5,0.884612,0.017635,0.883389,0.845464,"{'classifier__n_estimators': 200, 'classifier_..."
13,6,0.883002,0.014119,0.881118,0.848798,"{'classifier__n_estimators': 500, 'classifier_..."
3,7,0.879783,0.010973,0.870264,0.835738,"{'classifier__n_estimators': 300, 'classifier_..."
9,8,0.879383,0.011053,0.871461,0.855464,"{'classifier__n_estimators': 300, 'classifier_..."
20,9,0.879051,0.011340,0.872700,0.842240,"{'classifier__n_estimators': 200, 'classifier_..."
7,10,0.877776,0.011970,0.866000,0.816011,"{'classifier__n_estimators': 300, 'classifier_..."


Ausgewählte Konfiguration: Parametersuche


### Entscheidungsschwellen anhand der Validierung wählen

Für beide Konfigurationen wird die Schwelle mit der kleinsten False-Positive-Rate unter der Recall-Vorgabe gewählt. Bei gleichen Ergebnissen wird die größere Schwelle bevorzugt. Die Validierungswerte nach der Schwellenwahl sind ebenfalls Auswahlwerte; die unabhängige Bewertung folgt auf dem Testdatensatz. Die Schwellen sind modellspezifisch.


In [37]:
def rf_metrics(y_true, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "Schwelle": float(threshold), "TN": int(tn), "FP": int(fp),
        "FN": int(fn), "TP": int(tp),
        "Recall": tp / (tp + fn) if tp + fn else np.nan,
        "False-Negative-Rate": fn / (tp + fn) if tp + fn else np.nan,
        "False-Positive-Rate": fp / (fp + tn) if fp + tn else np.nan,
        "Precision": tp / (tp + fp) if tp + fp else np.nan,
        "Accuracy": (tp + tn) / (tp + tn + fp + fn),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
    }

def rf_choose_threshold(y_true, probabilities):
    # Alle unterschiedlichen Vorhersagewerte statt nur eines groben Rasters prüfen.
    thresholds = np.unique(np.r_[0.0, probabilities, 1.0])
    table = pd.DataFrame([rf_metrics(y_true, probabilities, t) for t in thresholds])
    eligible = table.loc[table["Recall"] >= RECALL_TARGET]
    selected = eligible.sort_values(
        ["False-Positive-Rate", "Schwelle"], ascending=[True, False]
    ).iloc[0]
    return float(selected["Schwelle"]), table

rf_val_probabilities = rf_validation_model.predict_proba(X_rf_val)[:, 1]
rf_baseline_val_probabilities = rf_baseline_validation.predict_proba(X_rf_val)[:, 1]
rf_threshold, rf_threshold_results = rf_choose_threshold(y_rf_val, rf_val_probabilities)
rf_baseline_threshold, rf_baseline_threshold_results = rf_choose_threshold(
    y_rf_val, rf_baseline_val_probabilities
)
print(f"Recall-Vorgabe für die Validierung: {RECALL_TARGET:.0%}")
print(f"Gewählte Schwelle: {rf_threshold:.4f}; Baseline-Schwelle: {rf_baseline_threshold:.4f}")
display(pd.DataFrame([
    {"Modell": "Baseline", **rf_metrics(y_rf_val, rf_baseline_val_probabilities, rf_baseline_threshold)},
    {"Modell": "Ausgewählter Random Forest", **rf_metrics(y_rf_val, rf_val_probabilities, rf_threshold)},
]))


Recall-Vorgabe für die Validierung: 95%
Gewählte Schwelle: 0.3245; Baseline-Schwelle: 0.2907


,Modell,Schwelle,TN,FP,FN,TP,Recall,False-Negative-Rate,False-Positive-Rate,Precision,Accuracy,ROC-AUC
0,Baseline,0.290721,47,35,5,97,0.95098,0.04902,0.426829,0.734848,0.782609,0.887733
1,Ausgewählter Random Forest,0.324496,50,32,5,97,0.95098,0.04902,0.390244,0.751938,0.798913,0.887733


### Mit festgelegten Einstellungen trainieren und Testdaten auswerten

Beide Modelle werden jetzt auf allen Trainingsdaten trainiert. Die vorher gewählten Schwellen bleiben fest; sie werden nicht an die Testwerte angepasst. Durch das erneute Training und Stichprobenschwankungen muss der Test-Recall nicht die 95-%-Vorgabe erreichen.

Der Testdatensatz wurde in früheren Notebook-Abschnitten bereits betrachtet. Für eine abschließende Aussage zur Übertragbarkeit sind zusätzliche, bisher ungenutzte Daten nötig.


In [38]:
rf_optimized = clone(rf_validation_model).fit(X_train, y_train)
rf_baseline_final = clone(rf_baseline).fit(X_train, y_train)
rf_test_probabilities = rf_optimized.predict_proba(X_test)[:, 1]
rf_baseline_test_probabilities = rf_baseline_final.predict_proba(X_test)[:, 1]
rf_test_comparison = pd.DataFrame([
    {"Modell": "Baseline, Schwelle 0.5", **rf_metrics(y_test, rf_baseline_test_probabilities, 0.5)},
    {"Modell": "Ausgewählt, Schwelle 0.5", **rf_metrics(y_test, rf_test_probabilities, 0.5)},
    {"Modell": "Baseline, Validierungsschwelle", **rf_metrics(y_test, rf_baseline_test_probabilities, rf_baseline_threshold)},
    {"Modell": "Ausgewählt, Validierungsschwelle", **rf_metrics(y_test, rf_test_probabilities, rf_threshold)},
])
display(rf_test_comparison)
rf_predictions = pd.DataFrame({
    "target": y_test,
    "prediction": (rf_test_probabilities >= rf_threshold).astype(int),
    "probability_heart_disease": rf_test_probabilities,
})
display(rf_predictions.head(10))
print("Für weitere bereinigte Daten:")
print("probabilities = rf_optimized.predict_proba(new_X[feature_columns])[:, 1]")
print("predictions = (probabilities >= rf_threshold).astype(int)")


,Modell,Schwelle,TN,FP,FN,TP,Recall,False-Negative-Rate,False-Positive-Rate,Precision,Accuracy,ROC-AUC
0,"Baseline, Schwelle 0.5",0.500000,65,17,9,93,0.911765,0.088235,0.207317,0.845455,0.858696,0.926829
1,"Ausgewählt, Schwelle 0.5",0.500000,64,18,10,92,0.901961,0.098039,0.219512,0.836364,0.847826,0.927786
2,"Baseline, Validierungsschwelle",0.290721,51,31,2,100,0.980392,0.019608,0.378049,0.763359,0.820652,0.926829
3,"Ausgewählt, Validierungsschwelle",0.324496,52,30,2,100,0.980392,0.019608,0.365854,0.769231,0.826087,0.927786


,target,prediction,probability_heart_disease
513,1,1,0.417348
824,1,1,0.901034
851,1,1,0.669306
803,1,1,0.911834
886,0,1,0.621757
483,0,1,0.438350
273,0,1,0.342860
539,1,1,0.933669
402,0,0,0.126311
600,1,1,0.655660


Für weitere bereinigte Daten:
probabilities = rf_optimized.predict_proba(new_X[feature_columns])[:, 1]
predictions = (probabilities >= rf_threshold).astype(int)


## Modelle nach Fehlalarmrate bei hohem Recall auswählen

Dieser Abschnitt ersetzt für die neue Auswahl die bisherige Optimierung nach Gesamt-ROC-AUC. **Experimentziel: 95 % Recall**, konfigurierbar über `COMPARISON_RECALL_TARGET`; keine klinische Vorgabe.

Für jedes Modell gibt es drei äußere Validierungsfolds innerhalb von `X_train`. Pro äußerem Fold werden Parameter ausschließlich auf dessen Trainingsanteil mit drei inneren Folds ausgewählt. Die Suchmetrik ist die maximal erreichbare Spezifität bei mindestens dem Ziel-Recall (Spezifität = 1 − Fehlalarmrate). Danach wird die Entscheidungsschwelle aus inneren Out-of-Fold-Vorhersagen festgelegt. Erst anschließend wird auf dem äußeren, bis dahin ungenutzten Fold bewertet.

Die Suche darf zur Berechnung ihrer Auswahlmetrik Schwellen auf inneren Validierungsfolds vergleichen. Diese Suchwerte und die inneren Out-of-Fold-Werte nach Parameterwahl sind keine unabhängige Leistungsbewertung. Dafür dienen die äußeren Folds. Der äußere Recall kann die Zielvorgabe unterschreiten.

Die bisherigen Testdaten werden in diesem Abschnitt nicht benutzt. Auch die äußeren Ergebnisse dienen letztlich der Modellauswahl; eine unabhängige Abschlussbewertung erfordert weitere bisher ungenutzte Daten. Drei äußere Folds geben nur eine erste Einschätzung der Stabilität.


In [39]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, roc_curve, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from joblib import parallel_backend

COMPARISON_RECALL_TARGET = 0.95
assert 0 < COMPARISON_RECALL_TARGET <= 1

def specificity_at_target_recall(y_true, probabilities, recall_target):
    fpr, tpr, _ = roc_curve(y_true, probabilities, drop_intermediate=False)
    return float(1 - fpr[tpr >= recall_target].min())

recall_constrained_scorer = make_scorer(
    specificity_at_target_recall,
    response_method="predict_proba",
    recall_target=COMPARISON_RECALL_TARGET,
)

def choose_oof_threshold(y_true, probabilities, recall_target):
    fpr, tpr, thresholds = roc_curve(y_true, probabilities, drop_intermediate=False)
    eligible = np.flatnonzero((tpr >= recall_target) & np.isfinite(thresholds))
    # roc_curve liefert Schwellen absteigend: bei gleicher FPR die höchste wählen.
    selected = eligible[np.argmin(fpr[eligible])]
    return float(thresholds[selected])

comparison_specs = {
    "Logistische Regression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"classifier__C": [0.01, 0.1, 1, 10]},
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
        {"classifier__max_depth": [6, None],
         "classifier__min_samples_leaf": [3, 5],
         "classifier__max_features": ["sqrt", 0.5]},
    ),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(
            max_iter=150, learning_rate=0.05, early_stopping=False,
            random_state=RANDOM_STATE,
        ),
        {"classifier__max_leaf_nodes": [7, 15],
         "classifier__l2_regularization": [0.0, 1.0]},
    ),
}

def make_comparison_search(name):
    classifier, grid = comparison_specs[name]
    pipeline = Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("classifier", clone(classifier)),
    ])
    return GridSearchCV(
        pipeline, grid, scoring=recall_constrained_scorer,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        refit=True, n_jobs=2, pre_dispatch=2, error_score="raise",
    )


In [40]:
outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + 1)
comparison_fold_rows = []
comparison_oof = pd.DataFrame(index=X_train.index)
comparison_outer_indices = []

with parallel_backend("threading", n_jobs=2):
    for fold, (fit_positions, val_positions) in enumerate(outer_cv.split(X_train, y_train), 1):
        X_outer_fit, X_outer_val = X_train.iloc[fit_positions], X_train.iloc[val_positions]
        y_outer_fit, y_outer_val = y_train.iloc[fit_positions], y_train.iloc[val_positions]
        comparison_outer_indices.append((X_outer_fit.index.tolist(), X_outer_val.index.tolist()))
        for name in comparison_specs:
            search_for_fold = make_comparison_search(name)
            search_for_fold.fit(X_outer_fit, y_outer_fit)
            # Diese Vorhersagen stammen jeweils von Modellen ohne die betreffende Zeile.
            inner_probabilities = cross_val_predict(
                clone(search_for_fold.best_estimator_), X_outer_fit, y_outer_fit,
                cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + 2),
                method="predict_proba", n_jobs=2,
            )[:, 1]
            threshold_for_fold = choose_oof_threshold(
                y_outer_fit, inner_probabilities, COMPARISON_RECALL_TARGET
            )
            outer_probabilities = search_for_fold.best_estimator_.predict_proba(X_outer_val)[:, 1]
            outer_predictions = (outer_probabilities >= threshold_for_fold).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_outer_val, outer_predictions, labels=[0, 1]).ravel()
            comparison_oof.loc[X_outer_val.index, name + "__probability"] = outer_probabilities
            comparison_oof.loc[X_outer_val.index, name + "__prediction"] = outer_predictions
            comparison_fold_rows.append({
                "Modell": name, "Fold": fold, "Schwelle": threshold_for_fold,
                "Recall": tp / (tp + fn), "Fehlalarmrate": fp / (fp + tn),
                "FN": int(fn), "FP": int(fp), "TN": int(tn), "TP": int(tp),
                "ROC-AUC": roc_auc_score(y_outer_val, outer_probabilities),
                "Parameter": search_for_fold.best_params_,
            })
            print(f"Fold {fold}: {name} — Recall {tp / (tp + fn):.1%}, Fehlalarmrate {fp / (fp + tn):.1%}")

comparison_folds = pd.DataFrame(comparison_fold_rows)
comparison_summary = comparison_folds.groupby("Modell").agg(
    Recall_Mittel=("Recall", "mean"), Recall_Std=("Recall", "std"),
    Fehlalarmrate_Mittel=("Fehlalarmrate", "mean"), Fehlalarmrate_Std=("Fehlalarmrate", "std"),
    FN_Gesamt=("FN", "sum"), FP_Gesamt=("FP", "sum"),
    ROC_AUC_Mittel=("ROC-AUC", "mean"),
)
comparison_summary["Recall_Ziel_erreicht"] = comparison_summary["Recall_Mittel"] >= COMPARISON_RECALL_TARGET
comparison_summary = comparison_summary.sort_values(
    ["Recall_Ziel_erreicht", "Fehlalarmrate_Mittel"], ascending=[False, True]
)
display(comparison_summary)
display(comparison_folds)


Fold 1: Logistische Regression — Recall 97.8%, Fehlalarmrate 51.4%
Fold 1: Random Forest — Recall 94.9%, Fehlalarmrate 45.0%
Fold 1: Gradient Boosting — Recall 96.3%, Fehlalarmrate 47.7%
Fold 2: Logistische Regression — Recall 94.1%, Fehlalarmrate 42.7%
Fold 2: Random Forest — Recall 94.8%, Fehlalarmrate 44.5%
Fold 2: Gradient Boosting — Recall 94.1%, Fehlalarmrate 48.2%
Fold 3: Logistische Regression — Recall 95.6%, Fehlalarmrate 43.1%
Fold 3: Random Forest — Recall 97.0%, Fehlalarmrate 47.7%
Fold 3: Gradient Boosting — Recall 98.5%, Fehlalarmrate 55.0%


,Recall_Mittel,Recall_Std,Fehlalarmrate_Mittel,Fehlalarmrate_Std,FN_Gesamt,FP_Gesamt,ROC_AUC_Mittel,Recall_Ziel_erreicht
Modell,,,,,,,,
Random Forest,0.955683,0.012721,0.457353,0.017192,18,150,0.891610,True
Logistische Regression,0.958079,0.018728,0.457409,0.048842,17,150,0.897002,True
Gradient Boosting,0.963054,0.022223,0.503114,0.041071,15,165,0.886852,True


,Modell,Fold,Schwelle,Recall,Fehlalarmrate,FN,FP,TN,TP,ROC-AUC,Parameter
0,Logistische Regression,1,0.184915,0.977941,0.513761,3,56,53,133,0.909066,{'classifier__C': 10}
1,Random Forest,1,0.317137,0.948529,0.449541,7,49,60,129,0.902590,"{'classifier__max_depth': None, 'classifier__m..."
2,Gradient Boosting,1,0.204099,0.963235,0.477064,5,52,57,131,0.902928,"{'classifier__l2_regularization': 1.0, 'classi..."
3,Logistische Regression,2,0.292018,0.940741,0.427273,8,47,63,127,0.894478,{'classifier__C': 0.1}
4,Random Forest,2,0.326969,0.948148,0.445455,7,49,61,128,0.892525,"{'classifier__max_depth': None, 'classifier__m..."
5,Gradient Boosting,2,0.170228,0.940741,0.481818,8,53,57,127,0.885253,"{'classifier__l2_regularization': 1.0, 'classi..."
6,Logistische Regression,3,0.146710,0.955556,0.431193,6,47,62,129,0.887462,{'classifier__C': 10}
7,Random Forest,3,0.291141,0.970370,0.477064,4,52,57,131,0.879715,"{'classifier__max_depth': None, 'classifier__m..."
8,Gradient Boosting,3,0.135122,0.985185,0.550459,2,60,49,133,0.872375,"{'classifier__l2_regularization': 0.0, 'classi..."


### Konfiguration und Schwelle für weitere Daten bereitstellen

Unter Modellen, deren mittlerer äußerer Recall das Ziel erreicht, wird die kleinste mittlere Fehlalarmrate gewählt. Erreicht keines das Ziel, wird das Modell mit dem höchsten mittleren Recall als vorläufiger Kandidat gewählt und ausdrücklich darauf hingewiesen. Diese Regel wird vor dem Betrachten der Ergebnisse festgelegt.

Für den Kandidaten werden Parameter und Schwelle nochmals ausschließlich mit den gesamten Trainingsdaten bestimmt. Das finale Modell wird auf allen Trainingsdaten trainiert. Die Schwelle basiert auf Out-of-Fold-Vorhersagen; für neue Daten wird sie nicht nachjustiert. Änderungen der Vorhersageverteilung durch das finale Training sind möglich. Die Zielvorgabe ist daher keine Garantie.


In [41]:
eligible_models = comparison_summary.loc[comparison_summary["Recall_Ziel_erreicht"]]
if len(eligible_models):
    comparison_selected_name = eligible_models["Fehlalarmrate_Mittel"].idxmin()
else:
    comparison_selected_name = comparison_summary.sort_values(
        ["Recall_Mittel", "Fehlalarmrate_Mittel"], ascending=[False, True]
    ).index[0]
    print("ACHTUNG: Kein Modell erreicht den Ziel-Recall im äußeren Mittel. Auswahl ist vorläufig.")

with parallel_backend("threading", n_jobs=2):
    comparison_final_search = make_comparison_search(comparison_selected_name)
    comparison_final_search.fit(X_train, y_train)
    comparison_final_oof_probabilities = cross_val_predict(
        clone(comparison_final_search.best_estimator_), X_train, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2),
        method="predict_proba", n_jobs=2,
    )[:, 1]
comparison_threshold = choose_oof_threshold(
    y_train, comparison_final_oof_probabilities, COMPARISON_RECALL_TARGET
)
comparison_model = comparison_final_search.best_estimator_
print("Vorläufig ausgewähltes Modell:", comparison_selected_name)
print("Parameter:", comparison_final_search.best_params_)
print(f"Festgelegte Schwelle: {comparison_threshold:.4f}")
print("Verwendung auf weiteren bereinigten Daten:")
print("p = comparison_model.predict_proba(new_X[feature_columns])[:, 1]")
print("prediction = (p >= comparison_threshold).astype(int)")
print("Die bisherigen Testdaten wurden in diesem Vergleich nicht verwendet.")


Vorläufig ausgewähltes Modell: Random Forest
Parameter: {'classifier__max_depth': 6, 'classifier__max_features': 0.5, 'classifier__min_samples_leaf': 3}
Festgelegte Schwelle: 0.2859
Verwendung auf weiteren bereinigten Daten:
p = comparison_model.predict_proba(new_X[feature_columns])[:, 1]
prediction = (p >= comparison_threshold).astype(int)
Die bisherigen Testdaten wurden in diesem Vergleich nicht verwendet.


In [1]:
def classify_with_uncertainty(model, X, lower=0.10, upper=0.70):
    if not 0 <= lower < upper <= 1:
        raise ValueError("Die Schwellen müssen 0 <= lower < upper <= 1 erfüllen.")

    p = model.predict_proba(X)[:, 1]

    status = np.select(
        [p < lower, p >= upper],
        ["Niedriger Modellwert", "Hoher Modellwert – Erkrankungsverdacht"],
        default="Unklar – weitere Untersuchung nötig",
    )

    return pd.DataFrame(
        {"Modellwert": p, "Ausgabe": status},
        index=X.index,
    )


# Drei Ausgaben auf den vorhandenen Testdaten; Grenzen sind nur Beispiele.
result = classify_with_uncertainty(comparison_model, X_test, lower=0.10, upper=0.70)
display(result)


,Modellwert,Ausgabe
513,0.425943,Unklar – weitere Untersuchung nötig
824,0.920055,Hoher Modellwert – Erkrankungsverdacht
851,0.687858,Unklar – weitere Untersuchung nötig
803,0.943481,Hoher Modellwert – Erkrankungsverdacht
886,0.580860,Unklar – weitere Untersuchung nötig
...,...,...
59,0.553336,Unklar – weitere Untersuchung nötig
814,0.904734,Hoher Modellwert – Erkrankungsverdacht
630,0.802035,Hoher Modellwert – Erkrankungsverdacht
739,0.762669,Hoher Modellwert – Erkrankungsverdacht


## Drei Gruppen auswerten: Fehler und Untersuchungsaufwand

Die folgenden Kennzahlen beziehen sich auf `result` und die zugehörigen Testlabels `y_test`. Die Grenzen 0,10 und 0,70 sind illustrative Werte, keine validierten Entscheidungskriterien. Sie werden in dieser Auswertung nicht optimiert.

Eine erkrankte Person in der niedrigen Gruppe ist ein übersehener Fall. Unklare Fälle werden separat gezählt, nicht als richtige Klassifikation. Wenn sowohl die hohe als auch die unklare Gruppe weitere Untersuchung erfordern, müssen beide Gruppen beim Untersuchungsaufwand und bei Weiterleitungen Nicht-Erkrankter berücksichtigt werden.

Die zusätzliche 95-%-Intervallschätzung zeigt Stichprobenunsicherheit unter binomialen Annahmen. Sie erfasst weder Modell-/Auswahlunsicherheit noch Unterschiede zwischen Kliniken. Null beobachtete Fehler garantieren keine Fehlerfreiheit. Die Testdaten wurden bereits mehrfach betrachtet; diese Auswertung ist explorativ.


In [2]:
from scipy.stats import binomtest

if not result.index.is_unique or not result.index.equals(y_test.index):
    raise ValueError("Ergebnisse und Testlabels müssen dieselben eindeutigen Indizes in derselben Reihenfolge haben.")
triage_analysis = result.join(y_test.rename("target"))
if triage_analysis["target"].isna().any():
    raise ValueError("Testlabels fehlen.")
triage_groups = [
    "Niedriger Modellwert",
    "Unklar – weitere Untersuchung nötig",
    "Hoher Modellwert – Erkrankungsverdacht",
]
if not triage_analysis["Ausgabe"].isin(triage_groups).all():
    raise ValueError("Unbekannte Ausgabegruppe.")
triage_counts = pd.crosstab(triage_analysis["Ausgabe"], triage_analysis["target"])
triage_counts = triage_counts.reindex(index=triage_groups, columns=[0, 1], fill_value=0)
triage_summary = triage_counts.rename(columns={0: "Nicht erkrankt", 1: "Erkrankt"})
triage_summary["Gesamt"] = triage_summary.sum(axis=1)
triage_summary["Anteil aller Fälle"] = triage_summary["Gesamt"] / len(triage_analysis)
triage_summary["Erkranktenanteil innerhalb der Gruppe"] = (
    triage_summary["Erkrankt"] / triage_summary["Gesamt"].replace(0, np.nan)
)
display(triage_summary)

low_negative, low_positive = triage_counts.loc[triage_groups[0], [0, 1]]
unclear_negative, unclear_positive = triage_counts.loc[triage_groups[1], [0, 1]]
high_negative, high_positive = triage_counts.loc[triage_groups[2], [0, 1]]
all_positive = int(triage_counts[1].sum())
all_negative = int(triage_counts[0].sum())
all_cases = len(triage_analysis)

metric_rows = [
    ("Erkrankte in der niedrigen Gruppe / alle Erkrankten", low_positive, all_positive),
    ("Erkrankte innerhalb der niedrigen Gruppe", low_positive, low_negative + low_positive),
    ("Nicht-Erkrankte in der hohen Gruppe / alle Nicht-Erkrankten", high_negative, all_negative),
    ("Nicht-Erkrankte in hoher oder unklarer Gruppe / alle Nicht-Erkrankten", high_negative + unclear_negative, all_negative),
    ("Hohe oder unklare Gruppe / alle Fälle (Untersuchungsaufwand)", all_cases - low_negative - low_positive, all_cases),
    ("Unklare Gruppe / alle Fälle", unclear_negative + unclear_positive, all_cases),
    ("Hohe oder niedrige Gruppe / alle Fälle (Abdeckung)", all_cases - unclear_negative - unclear_positive, all_cases),
    ("Erkrankte in hoher oder unklarer Gruppe / alle Erkrankten", high_positive + unclear_positive, all_positive),
    ("Erkrankte nur in der hohen Gruppe / alle Erkrankten", high_positive, all_positive),
]
triage_metrics = pd.DataFrame(metric_rows, columns=["Kennzahl", "Zähler", "Nenner"])
triage_metrics["Quote"] = triage_metrics["Zähler"] / triage_metrics["Nenner"].replace(0, np.nan)
display(triage_metrics)

for label, successes, total in [
    ("Erkranktenanteil in niedriger Gruppe", low_positive, low_negative + low_positive),
    ("Anteil übersehener Fälle unter allen Erkrankten", low_positive, all_positive),
]:
    if total:
        interval = binomtest(int(successes), int(total)).proportion_ci(confidence_level=0.95)
        print(f"{label}: {int(successes)}/{int(total)}; exaktes 95-%-Intervall {interval.low:.1%} bis {interval.high:.1%}")

assert triage_counts.to_numpy().sum() == len(y_test)
assert all_positive == int((y_test == 1).sum())
assert all_negative == int((y_test == 0).sum())


Erkranktenanteil in niedriger Gruppe: 0/22; exaktes 95-%-Intervall 0.0% bis 15.4%
Anteil übersehener Fälle unter allen Erkrankten: 0/102; exaktes 95-%-Intervall 0.0% bis 3.6%


target,Nicht erkrankt,Erkrankt,Gesamt,Anteil aller Fälle,Erkranktenanteil innerhalb der Gruppe
Ausgabe,,,,,
Niedriger Modellwert,22,0,22,0.119565,0.000000
Unklar – weitere Untersuchung nötig,53,29,82,0.445652,0.353659
Hoher Modellwert – Erkrankungsverdacht,7,73,80,0.434783,0.912500


,Kennzahl,Zähler,Nenner,Quote
0,Erkrankte in der niedrigen Gruppe / alle Erkra...,0,102,0.000000
1,Erkrankte innerhalb der niedrigen Gruppe,0,22,0.000000
2,Nicht-Erkrankte in der hohen Gruppe / alle Nic...,7,82,0.085366
3,Nicht-Erkrankte in hoher oder unklarer Gruppe ...,60,82,0.731707
4,Hohe oder unklare Gruppe / alle Fälle (Untersu...,162,184,0.880435
5,Unklare Gruppe / alle Fälle,82,184,0.445652
6,Hohe oder niedrige Gruppe / alle Fälle (Abdeck...,102,184,0.554348
7,Erkrankte in hoher oder unklarer Gruppe / alle...,102,102,1.000000
8,Erkrankte nur in der hohen Gruppe / alle Erkra...,73,102,0.715686


### Interpretation der aktuellen Ausgabe (Grenzen 0,10 und 0,70)

In der niedrigen Gruppe sind **22 Nicht-Erkrankte und keine Erkrankten**. Das ist ein gutes beobachtetes Ergebnis, aber eine kleine Gruppe: Das exakte 95-%-Intervall für ihren Erkranktenanteil reicht bei 0 von 22 bis ungefähr 15,4 %.

Die hohe Gruppe enthält **73 Erkrankte und 7 Nicht-Erkrankte**. Damit sind 91,3 % dieser Gruppe tatsächlich erkrankt. Die 7 eindeutigen Fehlalarme entsprechen 8,5 % aller Nicht-Erkrankten. Die hohe Gruppe allein erfasst allerdings nur 71,6 % der Erkrankten.

In der unklaren Gruppe befinden sich **29 Erkrankte und 53 Nicht-Erkrankte**. Das sind 44,6 % aller Fälle. Werden sowohl unklare als auch hohe Werte abgeklärt, werden hier sämtliche 102 Erkrankten weitergeleitet. Gleichzeitig benötigen 162 von 184 Personen (88,0 %) weitere Untersuchung, darunter 60 von 82 Nicht-Erkrankten (73,2 %).

**Die Dreiteilung reduziert die Zahl eindeutiger Fehlalarme, aber nicht automatisch die Zahl unnötiger Untersuchungen.** Gegenüber einer binären Entscheidung ist die niedrige eindeutige Fehlalarmrate deshalb keine alleinige Verbesserung. Beide Größen müssen zusammen mit den übersehenen Erkrankten und der unklaren Gruppe berichtet werden. Die Ergebnisse gelten für diese bereits betrachtete Stichprobe.


## Die verbleibenden hohen Fehlalarme untersuchen

Untersucht werden ausschließlich Fälle mit `target=0` und Ausgabe „Hoher Modellwert – Erkrankungsverdacht“ in der aktuellen Dreiteilung (`result`). Nicht-Erkrankte in der unklaren Gruppe sind zusätzliche Weiterleitungen und gehören nicht zu diesen sieben hohen Fehlalarmen.

Die folgenden Analysen sind explorativ, verändern weder Daten noch Modell noch Schwellen. Numerische Auffälligkeiten werden mit der **1,5-IQR-Regel anhand der beobachteten Trainingswerte** markiert. Diese statistische Regel ist kein medizinischer Grenzwert und erfasst keine mehrdimensionalen Ausreißer. Fehlende Werte werden separat gezählt.


In [2]:
if not result.index.equals(y_test.index):
    raise ValueError("Vorhersagen und Testlabels stimmen nicht überein.")
fp_high_indices = result.index[
    (result["Ausgabe"] == "Hoher Modellwert – Erkrankungsverdacht") & (y_test == 0)
]
if not len(fp_high_indices):
    raise ValueError("Keine hohen Fehlalarme in der aktuellen Ausgabe.")

fp_high_features = X_test.loc[fp_high_indices].copy()
fp_high_cases = df.loc[fp_high_indices, ["id", "dataset", "num", "target"] + feature_columns].copy()
fp_high_cases["Modellwert"] = result.loc[fp_high_indices, "Modellwert"]
fp_high_cases["Fehlende Merkmale"] = fp_high_features.isna().sum(axis=1)

fp_iqr_bounds = pd.DataFrame({
    "Q1": X_train[numeric_columns].quantile(0.25),
    "Q3": X_train[numeric_columns].quantile(0.75),
})
fp_iqr_bounds["Untergrenze"] = fp_iqr_bounds["Q1"] - 1.5 * (fp_iqr_bounds["Q3"] - fp_iqr_bounds["Q1"])
fp_iqr_bounds["Obergrenze"] = fp_iqr_bounds["Q3"] + 1.5 * (fp_iqr_bounds["Q3"] - fp_iqr_bounds["Q1"])
fp_iqr_flags = (
    fp_high_features[numeric_columns].lt(fp_iqr_bounds["Untergrenze"], axis="columns")
    | fp_high_features[numeric_columns].gt(fp_iqr_bounds["Obergrenze"], axis="columns")
)
fp_high_cases["IQR-Auffälligkeiten"] = fp_iqr_flags.apply(
    lambda row: ", ".join(row.index[row]) or "keine", axis=1
)
display(fp_high_cases.sort_values("Modellwert", ascending=False))
display(fp_iqr_bounds)

fp_numeric_comparison = pd.DataFrame({
    "Fehlalarme: Median": fp_high_features[numeric_columns].median(),
    "Training target=0: Median": X_train.loc[y_train == 0, numeric_columns].median(),
    "Training target=1: Median": X_train.loc[y_train == 1, numeric_columns].median(),
    "Fehlalarme: Minimum": fp_high_features[numeric_columns].min(),
    "Fehlalarme: Maximum": fp_high_features[numeric_columns].max(),
})
display(fp_numeric_comparison)

fp_missing_comparison = pd.DataFrame({
    "Fehlalarme: fehlend Anzahl": fp_high_features.isna().sum(),
    "Fehlalarme: fehlend Anteil": fp_high_features.isna().mean(),
    "Training target=0: fehlend Anteil": X_train.loc[y_train == 0].isna().mean(),
    "Training target=1: fehlend Anteil": X_train.loc[y_train == 1].isna().mean(),
})
display(fp_missing_comparison)

fp_category_rows = []
for column in categorical_columns:
    fp_values = fp_high_features[column].fillna("missing").astype(str)
    training_values = X_train[column].fillna("missing").astype(str)
    for value in sorted(fp_values.unique()):
        fp_category_rows.append({
            "Merkmal": column, "Wert": value,
            "Fehlalarme: Anzahl": int((fp_values == value).sum()),
            "Fehlalarme: Anteil": float((fp_values == value).mean()),
            "Training target=0: Anteil": float((training_values.loc[y_train == 0] == value).mean()),
            "Training target=1: Anteil": float((training_values.loc[y_train == 1] == value).mean()),
        })
fp_category_comparison = pd.DataFrame(fp_category_rows)
display(fp_category_comparison)

fp_site_comparison = pd.DataFrame({
    "Training: Fälle": df.loc[X_train.index].groupby("dataset").size(),
    "Training: Anteil target=1": df.loc[X_train.index].groupby("dataset")["target"].mean(),
    "Test: Fälle target=0": df.loc[y_test.index[y_test == 0]].groupby("dataset").size(),
    "Hohe Fehlalarme": fp_high_cases.groupby("dataset").size(),
}).fillna(0)
fp_site_comparison["Hohe Fehlalarme / Test target=0"] = (
    fp_site_comparison["Hohe Fehlalarme"] / fp_site_comparison["Test: Fälle target=0"].replace(0, np.nan)
)
display(fp_site_comparison)

# Technische Plausibilitätsprüfung der IDs und Original-Zielwerte.
fp_original_rows = pd.read_csv(ROOT / "data/heart_disease_uci.csv").set_index("id")
assert fp_high_cases["target"].eq(0).all()
assert fp_high_cases["num"].eq(0).all()
assert fp_original_rows.loc[fp_high_cases["id"], "num"].eq(0).all()
assert fp_high_cases["id"].is_unique
print(f"{len(fp_high_cases)} hohe Fehlalarme; ursprüngliche Zielwerte stimmen mit der bereinigten Tabelle überein.")


7 hohe Fehlalarme; ursprüngliche Zielwerte stimmen mit der bereinigten Tabelle überein.


,id,dataset,num,target,age,trestbps,chol,thalch,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,Modellwert,Fehlende Merkmale,IQR-Auffälligkeiten
467,469,Hungary,0,0,55.0,140.0,229.0,110.0,0.5,Male,asymptomatic,false,normal,true,flat,<NA>,<NA>,0.927107,2,keine
916,919,VA Long Beach,0,0,58.0,NaN,385.0,NaN,NaN,Male,asymptomatic,true,lv hypertrophy,<NA>,<NA>,<NA>,<NA>,0.849530,7,chol
487,489,Hungary,0,0,61.0,125.0,292.0,115.0,0.0,Male,asymptomatic,false,st-t abnormality,true,<NA>,<NA>,<NA>,0.832757,3,keine
409,411,Hungary,0,0,49.0,120.0,297.0,132.0,1.0,Male,asymptomatic,<NA>,normal,false,flat,<NA>,<NA>,0.812080,3,keine
412,414,Hungary,0,0,50.0,120.0,328.0,110.0,1.0,Female,asymptomatic,false,normal,true,flat,<NA>,<NA>,0.800407,2,keine
823,825,VA Long Beach,0,0,64.0,130.0,223.0,128.0,0.5,Male,asymptomatic,false,st-t abnormality,false,flat,<NA>,<NA>,0.795373,2,keine
482,484,Hungary,0,0,59.0,130.0,318.0,120.0,1.0,Male,non-anginal,false,normal,true,flat,<NA>,normal,0.749457,1,keine


,Q1,Q3,Untergrenze,Obergrenze
age,47.0,60.0,27.50,79.50
trestbps,120.0,140.0,90.00,170.00
chol,209.0,278.0,105.50,381.50
thalch,120.0,157.0,64.50,212.50
oldpeak,0.0,1.5,-2.25,3.75


,Fehlalarme: Median,Training target=0: Median,Training target=1: Median,Fehlalarme: Minimum,Fehlalarme: Maximum
age,58.00,51.0,57.0,49.0,64.0
trestbps,127.50,130.0,130.0,120.0,140.0
chol,297.00,233.0,247.5,223.0,385.0
thalch,117.50,151.0,127.0,110.0,132.0
oldpeak,0.75,0.0,1.0,0.0,1.0


,Fehlalarme: fehlend Anzahl,Fehlalarme: fehlend Anteil,Training target=0: fehlend Anteil,Training target=1: fehlend Anteil
age,0,0.000000,0.000000,0.000000
trestbps,1,0.142857,0.051829,0.081281
chol,0,0.000000,0.106707,0.310345
thalch,1,0.142857,0.051829,0.073892
oldpeak,1,0.142857,0.054878,0.081281
sex,0,0.000000,0.000000,0.000000
cp,0,0.000000,0.000000,0.000000
fbs,1,0.142857,0.033537,0.145320
restecg,0,0.000000,0.000000,0.000000
exang,1,0.142857,0.051829,0.073892


,Merkmal,Wert,Fehlalarme: Anzahl,Fehlalarme: Anteil,Training target=0: Anteil,Training target=1: Anteil
0,sex,Female,1,0.142857,0.353659,0.103448
1,sex,Male,6,0.857143,0.646341,0.896552
2,cp,asymptomatic,6,0.857143,0.256098,0.758621
3,cp,non-anginal,1,0.142857,0.320122,0.155172
4,fbs,false,5,0.714286,0.853659,0.684729
5,fbs,missing,1,0.142857,0.033537,0.145320
6,fbs,true,1,0.142857,0.112805,0.169951
7,restecg,lv hypertrophy,1,0.142857,0.201220,0.219212
8,restecg,normal,4,0.571429,0.652439,0.541872
9,restecg,st-t abnormality,2,0.285714,0.146341,0.238916


,Training: Fälle,Training: Anteil target=1,Test: Fälle target=0,Hohe Fehlalarme,Hohe Fehlalarme / Test target=0
dataset,,,,,
Cleveland,249,0.453815,29.0,0.0,0.000000
Hungary,228,0.368421,42.0,5.0,0.119048
Switzerland,96,0.916667,0.0,0.0,NaN
VA Long Beach,161,0.751553,11.0,2.0,0.181818


### Lokale Modellbeiträge entlang der Baumwege

Für jeden Fall wird pro Baum verfolgt, wie sich der Anteil positiver Trainingslabels entlang seines Entscheidungspfads ändert. Diese Änderungen werden dem jeweiligen Splitmerkmal zugeordnet und über die Bäume gemittelt. One-Hot-Spalten werden zu ihrem ursprünglichen Merkmal zusammengefasst; Indikatoren für fehlende numerische Messwerte bleiben separat.

**Basiswert + Summe der Beiträge = Vorhersagewert** wird numerisch geprüft. Positive Beiträge erhöhen den Modellwert, negative senken ihn. Diese Zerlegung erklärt die tatsächlichen Baumwege relativ zum Wurzelwert. Sie ist keine SHAP-Analyse und keine kausale oder klinische Erklärung. Beiträge hängen insbesondere bei korrelierten Merkmalen von den gewählten Splitwegen ab.


In [3]:
fp_processor = comparison_model.named_steps["preprocessing"]
fp_forest = comparison_model.named_steps["classifier"]
if not isinstance(fp_forest, RandomForestClassifier):
    raise TypeError("Diese Baumweg-Zerlegung setzt einen RandomForestClassifier voraus.")
fp_transformed = fp_processor.transform(fp_high_features)
if hasattr(fp_transformed, "toarray"):
    fp_transformed = fp_transformed.toarray()
# sklearn verwendet float32 für die Baumvergleiche; denselben Datentyp nutzen.
fp_transformed = np.asarray(fp_transformed, dtype=np.float32)
fp_encoded_names = fp_processor.get_feature_names_out()
fp_contributions = np.zeros_like(fp_transformed, dtype=float)
fp_basis = 0.0
positive_class_index = int(np.flatnonzero(fp_forest.classes_ == 1)[0])

for estimator in fp_forest.estimators_:
    tree = estimator.tree_
    node_values = tree.value[:, 0, :]
    positive_fraction = node_values[:, positive_class_index] / node_values.sum(axis=1)
    fp_basis += positive_fraction[0] / len(fp_forest.estimators_)
    for row_index, values in enumerate(fp_transformed):
        node = 0
        while tree.children_left[node] != tree.children_right[node]:
            feature = tree.feature[node]
            child = tree.children_left[node] if values[feature] <= tree.threshold[node] else tree.children_right[node]
            fp_contributions[row_index, feature] += (
                positive_fraction[child] - positive_fraction[node]
            ) / len(fp_forest.estimators_)
            node = child

fp_model_probabilities = comparison_model.predict_proba(fp_high_features)[:, positive_class_index]
np.testing.assert_allclose(
    fp_basis + fp_contributions.sum(axis=1), fp_model_probabilities, atol=1e-10
)
np.testing.assert_allclose(
    fp_model_probabilities, result.loc[fp_high_indices, "Modellwert"].to_numpy(), atol=1e-10
)

def fp_original_feature(encoded_name):
    if encoded_name.startswith("numeric__missingindicator_"):
        return "fehlend: " + encoded_name.removeprefix("numeric__missingindicator_")
    if encoded_name.startswith("numeric__"):
        return encoded_name.removeprefix("numeric__")
    for column in sorted(categorical_columns, key=len, reverse=True):
        if encoded_name.startswith("categorical__" + column + "_"):
            return column
    raise ValueError(f"Nicht zugeordnetes Merkmal: {encoded_name}")

fp_contribution_table = pd.DataFrame(fp_contributions, index=fp_high_indices, columns=fp_encoded_names)
fp_grouped_contributions = fp_contribution_table.T.groupby(
    [fp_original_feature(name) for name in fp_encoded_names], sort=False
).sum().T
fp_top_drivers = []
for index in fp_high_indices:
    contributions = fp_grouped_contributions.loc[index]
    positive_drivers = contributions[contributions > 0].sort_values(ascending=False).head(3)
    negative_drivers = contributions[contributions < 0].sort_values().head(2)
    fp_top_drivers.append({
        "id": int(df.loc[index, "id"]),
        "Modellwert": float(result.loc[index, "Modellwert"]),
        "Stärkste erhöhende Beiträge": "; ".join(f"{name}: +{value:.3f}" for name, value in positive_drivers.items()),
        "Stärkste senkende Beiträge": "; ".join(f"{name}: {value:.3f}" for name, value in negative_drivers.items()),
    })
fp_top_driver_table = pd.DataFrame(fp_top_drivers).sort_values("Modellwert", ascending=False)
print(f"Mittlerer Wurzel-Basiswert: {fp_basis:.3f}")
display(fp_top_driver_table)
display(fp_grouped_contributions.assign(id=df.loc[fp_high_indices, "id"]).set_index("id"))
print("Baumweg-Beiträge rekonstruieren alle sieben Vorhersagewerte korrekt.")


Mittlerer Wurzel-Basiswert: 0.551
Baumweg-Beiträge rekonstruieren alle sieben Vorhersagewerte korrekt.


,id,Modellwert,Stärkste erhöhende Beiträge,Stärkste senkende Beiträge
1,469,0.927107,cp: +0.145; exang: +0.141; thalch: +0.026,fbs: -0.002; oldpeak: -0.002
5,919,0.849530,cp: +0.183; age: +0.037; exang: +0.030,slope: -0.033; fehlend: chol: -0.011
4,489,0.832757,exang: +0.155; cp: +0.149; age: +0.032,oldpeak: -0.054; slope: -0.040
3,411,0.812080,cp: +0.214; fbs: +0.084; slope: +0.058,exang: -0.116; fehlend: chol: -0.034
6,414,0.800407,exang: +0.147; cp: +0.144; thalch: +0.026,sex: -0.099; thal: -0.017
2,825,0.795373,cp: +0.209; slope: +0.080; age: +0.041,exang: -0.118; fbs: -0.026
0,484,0.749457,exang: +0.180; thalch: +0.113; age: +0.059,cp: -0.168; thal: -0.057


,age,trestbps,chol,thalch,oldpeak,fehlend: trestbps,fehlend: chol,fehlend: thalch,fehlend: oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal
id,,,,,,,,,,,,,,,,,
484,0.058858,0.001328,0.018697,0.112642,-0.013828,0.000021,-0.008327,0.001379,0.000000,0.026017,-0.168436,0.002244,-0.009009,0.180002,0.039603,0.013796,-0.056900
469,0.009017,0.001979,0.002640,0.025968,-0.001891,0.000000,-0.001583,0.001146,0.000000,0.013376,0.145322,-0.002172,0.003440,0.140856,0.016206,0.016432,0.005003
825,0.040943,0.002573,0.000619,0.022845,0.005760,-0.000731,-0.020486,0.000099,0.000000,0.027611,0.208611,-0.026242,-0.004525,-0.118073,0.080265,0.035826,-0.011091
411,-0.024021,0.005516,0.003810,0.016728,0.013472,-0.000731,-0.034342,0.000225,0.000000,0.023263,0.213557,0.084033,-0.001939,-0.116257,0.058168,0.028614,-0.009385
489,0.031877,0.019013,-0.000311,0.020039,-0.054073,0.000000,-0.004832,0.001877,0.000000,0.010853,0.148637,-0.014862,-0.010210,0.154830,-0.039783,0.019070,-0.000737
919,0.037388,0.008954,0.005779,0.004429,0.010395,0.002059,-0.011100,-0.007342,-0.000218,0.013749,0.183455,0.027203,0.005778,0.029541,-0.032635,0.023117,-0.002391
414,-0.007022,-0.004019,0.014595,0.025834,0.009020,0.000000,-0.002214,0.000695,0.000000,-0.098752,0.143535,-0.007073,0.014620,0.147114,0.016867,0.012616,-0.016776


### Befunde für die aktuelle Stichprobe

**Keine auffällige Häufung extremer numerischer Werte:** Nur ID 919 hat eine IQR-Markierung: `chol=385` gegenüber einer oberen Trainings-IQR-Grenze von 381,5. Das ist eine knappe statistische Auffälligkeit, kein Nachweis eines Eingabefehlers. Für die anderen sechs Fälle liegt kein beobachteter numerischer Wert außerhalb dieser Grenzen. Mehrdimensionale Auffälligkeiten wurden damit nicht ausgeschlossen.

**Ähnliche Merkmalskombinationen wie in der positiven Klasse:** Sechs von sieben haben `cp=asymptomatic`, vier `exang=true`, fünf `slope=flat`. In den Trainingsdaten steht `cp=asymptomatic` bei 75,9 % der positiven gegenüber 25,6 % der negativen Fälle; `exang=true` bei 55,4 % gegenüber 13,4 %. Die beobachteten Herzfrequenzen `thalch` der Fehlalarme liegen bei 110–132, gegenüber Trainingsmedianen von 151 in der negativen und 127 in der positiven Klasse. Die Profile überschneiden sich mit der positiven Klasse, obwohl ihre Original-Zielwerte 0 sind.

**Tatsächliche Baumwege:** Bei sechs Fällen gehört `cp` zu den stärksten erhöhenden Beiträgen; beim siebten (ID 484) senkt `cp=non-anginal` den Wert, während `exang=true` und `thalch=120` stärker erhöhen. Bei ID 469 steigt der Wert vor allem durch `cp` (+0,145) und `exang` (+0,141) auf 0,927. Diese Beiträge sind Änderungen des Modellwerts entlang der Baumwege, keine medizinischen Effekte.

**Fehlende Informationen:** `ca` fehlt bei 7/7, `thal` bei 6/7. ID 919 hat sieben fehlende Modellmerkmale; ID 411 hat ein fehlendes `fbs`, dessen kodierter Modellbeitrag +0,084 beträgt. Die Beiträge von `ca` und `thal` sind insgesamt relativ klein; die Lücken allein erklären die Fehler daher nicht. Zum Vergleich fehlt `ca` auch bei 58,8 % der negativen Trainingsfälle und `thal` bei 52,4 %.

**Herkunft der Daten:** Fünf Fälle stammen aus Hungary, zwei aus VA Long Beach. Unter den negativen Testfällen entspricht das 5/42 beziehungsweise 2/11 hohen Fehlalarmen, gegenüber 0/29 in Cleveland. Die Stichproben sind klein. Unterschiedliche Merkmals-/Missingness-Muster und Klassenanteile zwischen Quellen sind mögliche Erklärungen; `dataset` selbst ist kein Modellmerkmal. Die Zuordnung zu einer Quelle beweist keine Ursache.

**Labels und Zieldefinition:** Für alle sieben stimmen `num=0` in Original- und bereinigter Datei überein; ein Label-Übertragungsfehler wurde nicht gefunden. Die UCI-Dokumentation beschreibt das ursprüngliche Ziel als angiografischen Erkrankungsstatus. Ein negatives Ziellabel bedeutet deshalb nicht, dass alle Symptome und Untersuchungsmerkmale unauffällig sein müssen. Aus diesen Angaben lässt sich kein fehlerhaftes Label oder eine alternative Diagnose feststellen. [UCI-Dokumentation](https://archive.ics.uci.edu/dataset/45/heart%2Bdisease)

Als nächste Trainings-/Validierungsanalyse bieten sich der Vergleich mit und ohne `ca`/`thal`, ein Vergleich der Missingness-Kodierung und eine Validierung nach Herkunftsklinik an. Verbesserungen sollten außerhalb dieser sieben bereits untersuchten Testfälle geprüft werden.


## Random Forest: mit und ohne `ca` / `thal`

Beide Varianten verwenden **dieselben drei äußeren Validierungsfolds innerhalb von `X_train`**, dieselbe Parametersuche und dasselbe Recall-Ziel wie der vorherige Modellvergleich. `ca` und `thal` werden gemeinsam entfernt, um den Einfluss dieser beiden stark unvollständigen Merkmale zu prüfen. Einzeleffekte werden damit nicht getrennt.

Pro äußerem Fold wird die Vorverarbeitung nur auf dessen Trainingsanteil gelernt. Dort wählen drei innere Folds die Parameter nach der Spezifität bei mindestens 95 % Recall. Eine zusätzliche innere Out-of-Fold-Vorhersage legt die modellspezifische Schwelle fest. Anschließend werden die äußeren Validierungsdaten mit diesen festgelegten Einstellungen ausgewertet. Der tatsächliche äußere Recall kann unter der Vorgabe liegen.

Die Testdaten und die sieben zuvor untersuchten Fehlalarme werden **nicht** für diesen Vergleich verwendet. Die beiden Varianten verändern weder `comparison_model` noch die bisherigen Ergebnisse. Benötigt werden die Zellen mit Dateneinlesen, Vorverarbeitung und den Hilfsfunktionen im Abschnitt „Modelle nach Fehlalarmrate bei hohem Recall auswählen“.


In [4]:
ABLATION_RECALL_TARGET = COMPARISON_RECALL_TARGET
ablation_variants = {
    "Mit ca und thal": categorical_columns.copy(),
    "Ohne ca und thal": [column for column in categorical_columns if column not in {"ca", "thal"}],
}
ablation_grid = {
    "classifier__max_depth": [6, None],
    "classifier__min_samples_leaf": [3, 5],
    "classifier__max_features": ["sqrt", 0.5],
}
ablation_scorer = make_scorer(
    specificity_at_target_recall, response_method="predict_proba",
    recall_target=ABLATION_RECALL_TARGET,
)

def make_ablation_search(categories):
    variant_preprocessor = ColumnTransformer([
        ("numeric", clone(numeric_pipeline), numeric_columns),
        ("categorical", clone(categorical_pipeline), categories),
    ], remainder="drop")
    variant_model = Pipeline([
        ("preprocessing", variant_preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=1
        )),
    ])
    return GridSearchCV(
        variant_model, ablation_grid, scoring=ablation_scorer,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        refit=True, n_jobs=2, pre_dispatch=2, error_score="raise",
    )

ablation_outer_splits = list(StratifiedKFold(
    n_splits=3, shuffle=True, random_state=RANDOM_STATE + 1
).split(X_train, y_train))
ablation_fold_rows = []
ablation_oof = pd.DataFrame(index=X_train.index)
ablation_feature_audit = []

with parallel_backend("threading", n_jobs=2):
    for fold, (fit_positions, val_positions) in enumerate(ablation_outer_splits, 1):
        for variant, categories in ablation_variants.items():
            variant_columns = numeric_columns + categories
            X_ab_fit = X_train.iloc[fit_positions][variant_columns]
            X_ab_val = X_train.iloc[val_positions][variant_columns]
            y_ab_fit, y_ab_val = y_train.iloc[fit_positions], y_train.iloc[val_positions]
            ab_search = make_ablation_search(categories)
            ab_search.fit(X_ab_fit, y_ab_fit)
            ab_inner_probabilities = cross_val_predict(
                clone(ab_search.best_estimator_), X_ab_fit, y_ab_fit,
                cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + 2),
                method="predict_proba", n_jobs=2,
            )[:, 1]
            ab_threshold = choose_oof_threshold(
                y_ab_fit, ab_inner_probabilities, ABLATION_RECALL_TARGET
            )
            ab_probabilities = ab_search.best_estimator_.predict_proba(X_ab_val)[:, 1]
            ab_predictions = (ab_probabilities >= ab_threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_ab_val, ab_predictions, labels=[0, 1]).ravel()
            ablation_oof.loc[X_ab_val.index, variant + "__probability"] = ab_probabilities
            ablation_oof.loc[X_ab_val.index, variant + "__prediction"] = ab_predictions
            ablation_fold_rows.append({
                "Variante": variant, "Fold": fold, "Schwelle": ab_threshold,
                "Recall": tp / (tp + fn), "Fehlalarmrate": fp / (fp + tn),
                "ROC-AUC": roc_auc_score(y_ab_val, ab_probabilities),
                "FN": int(fn), "FP": int(fp), "TN": int(tn), "TP": int(tp),
                "Parameter": ab_search.best_params_,
            })
            fitted_processor = ab_search.best_estimator_.named_steps["preprocessing"]
            fitted_names = fitted_processor.get_feature_names_out()
            ablation_feature_audit.append({
                "Variante": variant, "Fold": fold,
                "Eingabemerkmale": variant_columns.copy(),
                "Kodierte Merkmale": fitted_names.tolist(),
            })
            # Kontrolliert trainingsbasierte Imputation und echte Merkmalsentfernung.
            np.testing.assert_allclose(
                fitted_processor.named_transformers_["numeric"].named_steps["imputer"].statistics_,
                X_ab_fit[numeric_columns].median().fillna(0).to_numpy(),
            )
            assert set(X_ab_fit.index).isdisjoint(X_ab_val.index)
            if variant == "Ohne ca und thal":
                assert not {"ca", "thal"}.intersection(X_ab_fit.columns)
                assert not any(name.startswith(("categorical__ca_", "categorical__thal_")) for name in fitted_names)
            print(f"Fold {fold}: {variant} — Recall {tp / (tp + fn):.1%}, Fehlalarmrate {fp / (fp + tn):.1%}")

ablation_folds = pd.DataFrame(ablation_fold_rows)
ablation_summary = ablation_folds.groupby("Variante").agg(
    Recall_Mittel=("Recall", "mean"), Recall_Std=("Recall", "std"),
    Fehlalarmrate_Mittel=("Fehlalarmrate", "mean"), Fehlalarmrate_Std=("Fehlalarmrate", "std"),
    ROC_AUC_Mittel=("ROC-AUC", "mean"), FN_Gesamt=("FN", "sum"), FP_Gesamt=("FP", "sum"),
)
ablation_summary["Recall_Ziel_im_Mittel_erreicht"] = ablation_summary["Recall_Mittel"] >= ABLATION_RECALL_TARGET
display(ablation_summary)
display(ablation_folds)

assert ablation_oof.notna().all().all()
assert set(ablation_oof.index) == set(X_train.index)
assert set(ablation_oof.index).isdisjoint(X_test.index)
for variant in ablation_variants:
    counts = ablation_folds.loc[ablation_folds["Variante"] == variant, ["TN", "FP", "FN", "TP"]]
    assert int(counts.to_numpy().sum()) == len(y_train)
print("Vergleich und Prüfungen abgeschlossen; Testdaten wurden nicht ausgewertet.")


Fold 1: Mit ca und thal — Recall 94.9%, Fehlalarmrate 45.0%
Fold 1: Ohne ca und thal — Recall 94.9%, Fehlalarmrate 48.6%
Fold 2: Mit ca und thal — Recall 94.8%, Fehlalarmrate 44.5%
Fold 2: Ohne ca und thal — Recall 95.6%, Fehlalarmrate 41.8%
Fold 3: Mit ca und thal — Recall 97.0%, Fehlalarmrate 47.7%
Fold 3: Ohne ca und thal — Recall 97.0%, Fehlalarmrate 45.9%
Vergleich und Prüfungen abgeschlossen; Testdaten wurden nicht ausgewertet.


,Recall_Mittel,Recall_Std,Fehlalarmrate_Mittel,Fehlalarmrate_Std,ROC_AUC_Mittel,FN_Gesamt,FP_Gesamt,Recall_Ziel_im_Mittel_erreicht
Variante,,,,,,,,
Mit ca und thal,0.955683,0.012721,0.457353,0.017192,0.891610,18,150,True
Ohne ca und thal,0.958152,0.011150,0.454379,0.034235,0.883552,17,149,True


,Variante,Fold,Schwelle,Recall,Fehlalarmrate,ROC-AUC,FN,FP,TN,TP,Parameter
0,Mit ca und thal,1,0.317137,0.948529,0.449541,0.902590,7,49,60,129,"{'classifier__max_depth': None, 'classifier__m..."
1,Ohne ca und thal,1,0.324350,0.948529,0.486239,0.895642,7,53,56,129,"{'classifier__max_depth': 6, 'classifier__max_..."
2,Mit ca und thal,2,0.326969,0.948148,0.445455,0.892525,7,49,61,128,"{'classifier__max_depth': None, 'classifier__m..."
3,Ohne ca und thal,2,0.347397,0.955556,0.418182,0.881414,6,46,64,129,"{'classifier__max_depth': 6, 'classifier__max_..."
4,Mit ca und thal,3,0.291141,0.970370,0.477064,0.879715,4,52,57,131,"{'classifier__max_depth': None, 'classifier__m..."
5,Ohne ca und thal,3,0.299830,0.970370,0.458716,0.873598,4,50,59,131,"{'classifier__max_depth': 6, 'classifier__max_..."


### Unterschiede pro identischer Aufteilung

Positive Differenzen bedeuten höhere Werte **ohne** `ca`/`thal`: Bei Recall ist das günstig, bei Fehlalarmrate ungünstig. Alle Differenzen der Raten werden in Prozentpunkten berichtet. Mehrere Kennzahlen müssen gemeinsam gelesen werden: Eine geringere Fehlalarmrate bei gleichzeitig geringerem Recall ist kein eindeutiger Gewinn. Drei Folds erlauben keine sichere Aussage über kleine Unterschiede.


In [5]:
ablation_with = ablation_folds.loc[ablation_folds["Variante"] == "Mit ca und thal"].set_index("Fold")
ablation_without = ablation_folds.loc[ablation_folds["Variante"] == "Ohne ca und thal"].set_index("Fold")
ablation_paired_differences = pd.DataFrame({
    "Recall: ohne minus mit (Prozentpunkte)": 100 * (ablation_without["Recall"] - ablation_with["Recall"]),
    "Fehlalarmrate: ohne minus mit (Prozentpunkte)": 100 * (ablation_without["Fehlalarmrate"] - ablation_with["Fehlalarmrate"]),
    "ROC-AUC: ohne minus mit": ablation_without["ROC-AUC"] - ablation_with["ROC-AUC"],
    "FN: ohne minus mit": ablation_without["FN"] - ablation_with["FN"],
    "FP: ohne minus mit": ablation_without["FP"] - ablation_with["FP"],
})
display(ablation_paired_differences)
print("Mittlere Unterschiede:")
display(ablation_paired_differences.mean().to_frame("Ohne minus mit"))
print("Gesamt über alle äußeren Folds:")
print(f"Zusätzliche übersehene Erkrankte ohne ca/thal: {int(ablation_paired_differences['FN: ohne minus mit'].sum()):+d}")
print(f"Zusätzliche Fehlalarme ohne ca/thal: {int(ablation_paired_differences['FP: ohne minus mit'].sum()):+d}")


Mittlere Unterschiede:
Gesamt über alle äußeren Folds:
Zusätzliche übersehene Erkrankte ohne ca/thal: -1
Zusätzliche Fehlalarme ohne ca/thal: -1


,Recall: ohne minus mit (Prozentpunkte),Fehlalarmrate: ohne minus mit (Prozentpunkte),ROC-AUC: ohne minus mit,FN: ohne minus mit,FP: ohne minus mit
Fold,,,,,
1,0.000000,3.669725,-0.006948,0,4
2,0.740741,-2.727273,-0.011111,-1,-3
3,0.000000,-1.834862,-0.006116,0,-2


,Ohne minus mit
Recall: ohne minus mit (Prozentpunkte),0.246914
Fehlalarmrate: ohne minus mit (Prozentpunkte),-0.297470
ROC-AUC: ohne minus mit,-0.008059
FN: ohne minus mit,-0.333333
FP: ohne minus mit,-0.333333


### Ergebnis dieser Ausführung

Mit `ca` und `thal` beträgt der mittlere äußere Recall **95,6 %**, die mittlere Fehlalarmrate **45,7 %** und die mittlere ROC-AUC **0,892**. Ohne beide Merkmale ergeben sich **95,8 % Recall**, **45,4 % Fehlalarmrate** und **0,884 ROC-AUC**.

Über alle 734 jeweils einmal äußerlich validierten Trainingsfälle sinken die übersehenen Erkrankten von **18 auf 17** und die Fehlalarme von **150 auf 149**. Der Unterschied bei der Fehlalarmrate beträgt etwa **−0,30 Prozentpunkte**, beim Recall etwa **+0,25 Prozentpunkte**.

Das ist ein kleiner beobachteter Unterschied, kein überzeugender Nachweis einer Verbesserung. Im ersten Fold erzeugt die Variante ohne beide Merkmale vier zusätzliche Fehlalarme; in den anderen beiden Folds drei beziehungsweise zwei weniger. Die ROC-AUC ist ohne die Merkmale in allen drei Folds niedriger. Beide Varianten erfüllen das Recall-Ziel nur im Mittel, nicht in jedem Fold.

Das gemeinsame Entfernen von `ca` und `thal` bringt hier keinen deutlichen Fortschritt beim gewünschten Kompromiss. Die ursprüngliche Pipeline wird deshalb durch diesen Vergleich nicht automatisch ersetzt. Eine Entscheidung kann auch berücksichtigen, ob diese Untersuchungsmerkmale bei späteren Daten überhaupt verfügbar sind. Die Ergebnisse dienen der explorativen Auswahl und sind keine unabhängige Abschlussbewertung.
